For the purposes of demonstration, is it necessary to build a synthetic data set to anonymise any real data. The contents of this notebook generate a synthetic dataset in the same format as the real data set.

The data is intentionally basic with only simple trends, as this project is to develop a training tracker, not an advanced synthetic data generator. With the addition of future features, it may be necessary to make these functions more complex to maintain good demonstration of the features.

In [1]:
from datetime import datetime
import numpy as np
import pandas as pd

np.random.seed(42)

dates = pd.date_range("2020-01-01", "2025-12-31", freq="D")
n = len(dates)

output_dir = "../data/synthetic/"

# Generate Mock Data

## Weight

In [2]:
# Weight trend: gradual improvement with seasonal fluctuations
trend = np.linspace(100, 85, n)
season = 0.6*np.sin(np.linspace(0, 20*np.pi, n))
noise = np.random.normal(0, 0.35, n)
weight_kg = np.clip(trend + season + noise, 80, 110)

weight_df = pd.DataFrame({
    "datetime": dates.strftime("%Y-%m-%d 00:00:00"),
    "Weight": np.round(weight_kg, 1),
    "Fasting": ''
})

weight_path=f"{output_dir}mock_cronometer_weight.csv"

weight_df.to_csv(weight_path,index=False)

## Body Fat

In [3]:
# body fat trend: gradual improvement with seasonal fluctuations
trend = np.linspace(26.5, 15.5, n)
season = 0.6*np.sin(np.linspace(0, 20*np.pi, n))
noise = np.random.normal(0, 0.35, n)
bodyfat = np.clip(trend + season + noise, 10, 35)

bodyfat_df = pd.DataFrame({
    "DateTime": dates.strftime("%d/%m/%Y 06:30"),
    "Body Fat": np.round(bodyfat, 1),
    "Fasting": ""
})

bf_path=f"{output_dir}mock_cronometer_bodyfat.csv"

bodyfat_df.to_csv(bf_path,index=False)

## Heart Rate

In [4]:
# Resting heart rate: fitness improves over time
hr_trend = np.linspace(68, 53, n)
hr = np.round(hr_trend + 3*np.sin(np.linspace(0, 14*np.pi, n))
              + np.random.normal(0, 2.5, n)).astype(int)
hr = np.clip(hr, 45, 95)

heartrate_df = pd.DataFrame({
    "DateTime": dates.strftime("%d/%m/%Y 00:00"),
    "Heart Rate": hr,
    "Fasting": ""
})

hr_path=f"{output_dir}mock_cronometer_heartrate.csv"

heartrate_df.to_csv(hr_path,index=False)

## Food

In [5]:
# Macronutrients (kcal from macros to resemble Cronometer export)
training = (np.sin(np.linspace(0, 40*np.pi, n)) > 0).astype(int)
protein_g = np.clip(
    130 + 35*training + np.random.normal(0, 15, n), 60, 230
)
carbs_g = np.clip(
    280 + 120*training + np.random.normal(0, 45, n), 100, 600
)
fat_g = np.clip(
    70 + 15*np.random.randn(n), 25, 150
)
alcohol_g = np.where(
    np.random.rand(n) < 0.08,
    np.random.choice([23, 46, 115, 140], size=n),
    0
)

calories = (protein_g*4) + (carbs_g*4) + (fat_g*9) + (alcohol_g*7)

food = pd.DataFrame({
    "DateTime": dates.strftime("%d/%m/%Y 00:00"),
    "Alcohol": alcohol_g * 7.,
    "Fat": fat_g * 9.,
    "Carbs": carbs_g * 4.,
    "Protein": protein_g * 4.,
    "Fasting": ""
})

food_path=f"{output_dir}mock_cronometer_food.csv"

food.to_csv(food_path,index=False)

## Sleep

In [6]:
# Sleep: slight improvement over time
sleep_trend = np.linspace(7, 9, n)
sleep = np.round(sleep_trend + 3*np.sin(np.linspace(0, 14*np.pi, n))
              + np.random.normal(0, 2.5, n),1).astype(float)

sleep = np.clip(sleep, 4, 12)

sleep_df = pd.DataFrame({
    "DateTime": dates.strftime("%d/%m/%Y 00:00"),
    "Sleep": sleep,
    "Fasting": ""
})

sleep_path=f"{output_dir}mock_cronometer_sleep.csv"

sleep_df.to_csv(sleep_path,index=False)

## Training Log

In [7]:
from openpyxl import Workbook
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(7)

catalog={
"Squat":"S001","Bench Press":"B001","Deadlift":"D001","Overhead Press":"O001",
"Barbell Row":"R001","Pull Up":"PU01","Reverse Lunge":"L001","Leg Extension":"L002",
"Romanian Deadlift":"D002","Incline DB Press":"B002","Lat Pulldown":"PU02",
"Leg Curl":"L003","Bicep Curl":"A001","Tricep Pushdown":"A002"}

catalog_muscle_group = {
    "Squat":"hamstring, quad, glute",
    "Bench Press":"chest, shoulder, triceps",
    "Deadlift":"hamstring, back, glute",
    "Overhead Press":"shoulder, triceps",
    "Barbell Row":"back, biceps",
    "Pull Up":"back, biceps",
    "Reverse Lunge":"quad, hamstring, glute",
    "Leg Extension":"quad",
    "Romanian Deadlift":"hamstring, glute",
    "Incline DB Press":"chest, shoulder, triceps",
    "Lat Pulldown":"back",
    "Leg Curl":"hamstring",
    "Bicep Curl":"biceps",
    "Tricep Pushdown":"triceps"
}

templates={
"Lower":["Squat","Reverse Lunge","Leg Extension","Leg Curl"],
"Upper":["Bench Press","Barbell Row","Incline DB Press","Lat Pulldown","Bicep Curl","Tricep Pushdown"]}

strength={"Squat":100,"Bench Press":75,"Barbell Row":65,"Reverse Lunge":30,
"Leg Extension":90,"Leg Curl":50,"Incline DB Press":28,"Lat Pulldown":60,
"Bicep Curl":18,"Tricep Pushdown":35}

cardio = ["zone 2 run", "Threshold Run", "bike"]

start=datetime(2016,1,4); end=datetime(2025,12,31)
dates=[]
d=start
while d<=end:
    if d.weekday() in [0,1,3,4]:
        dates.append(d)
    d+=timedelta(days=1)

rows_strength=[]; 
rows_sessions=[]; 
for i,dt in enumerate(dates):
    sid=1
    prog=1+0.35*(i/len(dates))
    daytype="Lower" if i%2==0 else "Upper"
    session=dt.strftime("%Y-%m-%d")+("A")
    rows_sessions.append([session, dt, "strength",""])
    for ex in templates[daytype]:
        base=strength[ex]*prog*(1+np.random.normal(0,0.03))
        if ex in ["Squat","Bench Press"]:
            for frac,reps in zip([0.2,0.5,0.7,0.85],[10,5,5,3]):
                load=round((base*frac)/2.5)*2.5
                e1=round(load*(1+reps/30))
                rows_strength.append([sid,session,ex,catalog[ex],load,reps,"",e1]); sid+=1
            for _ in range(4):
                reps=int(np.random.choice([3,4,5,6]))
                load=round(base/2.5)*2.5
                e1=round(load*(1+reps/30))
                rows_strength.append([sid,session,ex,catalog[ex],load,reps,"",e1]); sid+=1
        else:
            for _ in range(np.random.randint(2,5)):
                reps=int(np.random.choice([8,10,12]))
                load=max(5,round(base/2)*2)
                e1=round(load*(1+reps/30))
                rows_strength.append([sid,session,ex,catalog[ex],load,reps,"",e1]); sid+=1

# df = pd.DataFrame(rows_strength,columns=["set_id","session_id","exercise_name","exercise_id","load_kg","reps","rpe","e1rm"])

strength_df = pd.DataFrame(rows_strength,columns=["set_id","session_id","exercise_name","exercise_id","load_kg","reps","rpe","e1rm"])

sessions_df = pd.DataFrame(rows_sessions,columns=["session_id", "date", "session_type", "notes"])

wb=Workbook()

# strengh_log
ws=wb.active
ws.title="strength_log"
ws.append(strength_df.columns.tolist())
for row in strength_df.itertuples(index=False):
    ws.append(list(row))

# session_log
ws2 = wb.create_sheet(title="sessions")
ws2.append(sessions_df.columns.tolist())
for row in sessions_df.itertuples(index=False):
    ws2.append(list(row))

# exercise type
rows_type = []; 
for x in catalog:
    rows_type.append([catalog[x], x, "", catalog_muscle_group[x]])
exercise_type_df = pd.DataFrame(rows_type, columns=["exercise_id", "exercise_name", "movement_pattern","muscle_group"])

ws3 = wb.create_sheet(title="exercise_type")
ws3.append(exercise_type_df.columns.tolist())
for row in exercise_type_df.itertuples(index=False):
    ws3.append(list(row))

xlsx_path=f"{output_dir}mock_training.xlsx"
wb.save(xlsx_path)